# TRAM Demo: ResNet18 on CIFAR-10

Reproduces the **ResNet18 / CIFAR-10 / w8a8** datapoint reported in the DAC'26 paper
*“TRAM: Training Approximate Multiplier Structures for Low-Power AI Accelerators”*
by Meng, Wang, Ye, Yu, Burleson, and De Micheli
([PDF](paper/Meng_Wang_Ye_Yu_Burleson_De%20Micheli_TRAM%20Training%20Approximate%20Multiplier%20Structures%20for%20Low-Power%20AI%20Accelerators.pdf)).

The notebook walks through the three TRAM phases end-to-end and is meant to guide
a reimplementation:

| Section | Phase / Step                                | Reproduces |
|---------|---------------------------------------------|------------|
| 0       | Setup, dataset, pretrained checkpoints      | —          |
| 1       | FP32 + w8a8 PTQ baselines                   | the FP32 row + the *AccMul* marker for ResNet18 in Figure 4 |
| 2       | Phase 1 — design-space exploration (DSE)    | a single (λ)-point on the TRAM curve in Figure 4 |
| 3       | Phase 2 — AxM structure mapping             | the Verilog/BLIF/LUT artifacts feeding phase 3 |
| 4       | Phase 3 — accuracy recovery                 | the final accuracy of the chosen (λ)-point |
| 5       | Bonus: reproduce paper Table 1 with the simulator | every numeric entry in Table 1 |

### Hardware note — laptop 4060 vs. paper A100

This notebook was tested on a **laptop with an NVIDIA RTX 4060 (8 GB)** GPU.
The paper experiments ran on a single **NVIDIA A100**. Even with the same
seed and the same checkpoint, **the numbers will not bit-match the paper's
reported numbers** because cuDNN can pick different algorithms / reduction
orders across GPU architectures:

- **§1 — released checkpoints, eval only.** Deterministic; should match the
  *AccMul* marker for ResNet18 in Figure 4 to very close accuracy on a 4060.
- **§2–§4 — TRAM phases 1, 2, 3.** Driven with the paper's 10-epoch setting
  for phase 1 and phase 3 (Section 6 of the paper). Wall-clock on a 4060 will
  be substantially longer than on an A100; the resulting λ-point should
  *land near* the corresponding TRAM marker in Figure 4 but with small
  numerical drift due to GPU-architecture differences.
- **§5 — Table 1 simulator metrics.** CPU-only, exhaustive simulation —
  reproduces every numeric entry of Table 1 **byte-for-byte** regardless of
  host hardware.

**Pretrained models** are downloaded automatically from the
[v1.0.0 release](https://github.com/changmg/TRAM/releases/tag/v1.0.0).

## 0. Setup

Update `CIFAR10_DIR` below to a writable path; CIFAR-10 will be auto-downloaded by torchvision on first use.

In [1]:
import os
import sys
import pathlib
import urllib.request

import torch

ROOT = pathlib.Path().resolve()
sys.path.insert(0, str(ROOT))

CIFAR10_DIR = os.environ.get("CIFAR10_DIR", "./data/cifar10")
PRETRAINED_DIR = ROOT / "pretrained"
PRETRAINED_DIR.mkdir(exist_ok=True)

# Every artifact produced by phases 1–3 (logs, the trained model, the Verilog,
# the BLIF, the LUT) lands here.
RESULTS = ROOT / "demo_resnet18_cifar10_results"
RESULTS.mkdir(exist_ok=True)

# Verify the CUDA self-ops extension is built. Phases 1 and 3 import
# `approx_ops` at module load time via `self_ops`; if the .so is missing the
# subprocess will fail with `ModuleNotFoundError: No module named 'approx_ops'`.
_so_files = sorted(ROOT.glob("approx_ops*.so"))
if not _so_files:
    raise RuntimeError(
        "CUDA self-ops extension not found in the project root.\n"
        "Build it once with:\n"
        "    python ./setup.py build_ext --inplace\n"
        "(see README.md → 'Dependencies & Build' → step 2 'CUDA self-ops')."
    )

# Override the dataset path baked into conf/global_vars.py for this session
import conf.global_vars
conf.global_vars.DATASET_PATHS["cifar10"] = CIFAR10_DIR
from conf import settings

from utils.common import set_seed
set_seed(1005)

print("torch:        ", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:          ", torch.cuda.get_device_name(0))
print("CUDA self-ops:", _so_files[0].name)
print("CIFAR-10 path:", CIFAR10_DIR)
print("results dir:  ", RESULTS)

torch:         2.3.0+cu121
CUDA available: True
GPU:           NVIDIA GeForce RTX 4060 Laptop GPU
CUDA self-ops: approx_ops.cpython-311-x86_64-linux-gnu.so
CIFAR-10 path: ./data/cifar10
results dir:   /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results


In [2]:
# Fetch the two ResNet18 checkpoints used in this notebook
RELEASE_BASE = "https://github.com/changmg/TRAM/releases/download/v1.0.0"
FP32_CKPT = PRETRAINED_DIR / "cifar10_resnet18_fp32.pth"
PTQ_CKPT  = PRETRAINED_DIR / "cifar10_resnet18_w8a8_ptq.pth"

for ckpt in (FP32_CKPT, PTQ_CKPT):
    if ckpt.exists():
        print(f"already present: {ckpt.name} ({ckpt.stat().st_size / 1e6:.1f} MB)")
        continue
    url = f"{RELEASE_BASE}/{ckpt.name}"
    print(f"downloading {url} → {ckpt}")
    urllib.request.urlretrieve(url, ckpt)
    print(f"  done ({ckpt.stat().st_size / 1e6:.1f} MB)")

already present: cifar10_resnet18_fp32.pth (44.8 MB)
already present: cifar10_resnet18_w8a8_ptq.pth (45.1 MB)


In [3]:
# Build CIFAR-10 train + test data loaders (test loader is what we need for eval)
import utils.datasets as datasets

train_loader, test_loader = datasets.cifar10.build_cifar10_data(
    data_path=CIFAR10_DIR, batch_size=256, workers=4)
print(f"train: {len(train_loader.dataset):,} images, test: {len(test_loader.dataset):,} images")

Files already downloaded and verified
Files already downloaded and verified
train: 50,000 images, test: 10,000 images


## 1. Baselines — FP32 and w8a8 PTQ

We first establish the two pre-AxM accuracy points that bound where TRAM
operates:

- **FP32 baseline.** The unquantized ResNet18 from
  `cifar10_resnet18_fp32.pth`. Expected ~94 %. This is the upper bound;
  everything below pays for either quantization or AxM approximation.
- **w8a8 PTQ baseline — the “AccMul” point in paper Figure 4.** We **load** the
  pre-quantized checkpoint `cifar10_resnet18_w8a8_ptq.pth` from the v1.0.0
  release; there is no calibration step in the notebook (PTQ has already been
  done). The configuration is 8-bit weights with channel-wise quantization
  and 8-bit activations with layer-wise quantization, evaluated with the
  **accurate multiplier** (no AxM substitution). This number should match the
  *AccMul* marker in the ResNet18 panel of Figure 4.

> *Optional — produce the PTQ checkpoint yourself.* The released
> `cifar10_resnet18_w8a8_ptq.pth` was generated by
> [`main_qat.py`](main_qat.py). To regenerate it from the FP32 baseline:
> ```bash
> python main_qat.py \
>     --pretrained_weights pretrained/cifar10_resnet18_fp32.pth \
>     --channel_wise --nbits_w 8 --nbits_a 8 \
>     --log demo_resnet18_cifar10_results/resnet18_w8a8_ptq.log
> ```

In [4]:
# Two baselines: FP32 ResNet18 and the released w8a8 PTQ checkpoint.
import models.cifar10
from utils.common import test_model
from quant.quant_model import QuantModel

# --- FP32 baseline ---
fp_model = models.cifar10.resnet.resnet18().cuda()
fp_model.load_state_dict(torch.load(FP32_CKPT, map_location="cuda", weights_only=True))
acc1_fp32, _ = test_model(fp_model, test_loader)
print(f"FP32 ResNet18 / CIFAR-10 top-1 accuracy: {acc1_fp32 * 100:.2f}%")
assert acc1_fp32 > 0.93, f"unexpected FP32 accuracy: {acc1_fp32:.4f}"

# --- w8a8 PTQ baseline (the "AccMul" point in paper Figure 4) ---
wq_params = {"n_bits": 8, "channel_wise": True}
aq_params = {"n_bits": 8, "channel_wise": False}
q_model = QuantModel(
    model=models.cifar10.resnet.resnet18(),
    dataset_name="cifar10",
    weight_quant_params=wq_params,
    act_quant_params=aq_params,
).cuda()
q_model.prepare_quantization_aware_training()
q_model.load_state_dict(torch.load(PTQ_CKPT, map_location="cuda", weights_only=True))
q_model.switch_train_eval_mode(train_mode=False)
acc1_w8a8, _ = test_model(q_model, test_loader)
print(f"w8a8 PTQ (AccMul) top-1 accuracy: {acc1_w8a8 * 100:.2f}%  ← compare with the AccMul marker in Figure 4")

Testing model: 100%|██████████| 40/40 [00:00<00:00, 42.48it/s]


FP32 ResNet18 / CIFAR-10 top-1 accuracy: 93.07%


Testing model: 100%|██████████| 40/40 [00:01<00:00, 24.47it/s]

w8a8 PTQ (AccMul) top-1 accuracy: 93.10%  ← compare with the AccMul marker in Figure 4


## 2. Phase 1 — Design-Space Exploration

Phase 1 jointly optimizes the AxM **structure parameters** $\Theta$ and the model
weights $\mathbf{W}$ on

$$
\min_{\Theta,\mathbf{W}}\; \mathcal{L}_{\text{power}}(\Theta)\cdot\lambda + \mathcal{L}_{\text{AI\_model}}(\Theta,\mathbf{W},\mathbf{X})
$$

(Eq. 2 of the paper). Each layer's AxM is parameterized by
$\Theta^{(l)}=[\theta_0^{(l)},\dots,\theta_{P-1}^{(l)}]$, where $\theta_c^{(l)}\in[0,1]$
is a *continuous* approximation degree for column $c$ (Section 3.2).

The cell below shells out to [`main_phase1.py`](main_phase1.py) with the
**paper-faithful settings** from Section 6: batch 256, SGD with momentum 0.9,
weight decay 5e-4, learning rate 5e-4, $P=4$, $\lambda=0.3$, **10 epochs**.
Make sure [`conf/global_vars.py`](conf/global_vars.py) points at your CIFAR-10
directory before running (the subprocess does **not** inherit the in-process
override done in §0).

In [5]:
# Phase 1 — invoke main_phase1.py as a subprocess (no in-notebook reimplementation).
import subprocess

LAMBDA = 0.3
EPOCHS_PHASE1 = 10  # paper Section 6 setting
PHASE1_LOG = RESULTS / f"resnet18_w8a8_lbd{LAMBDA}_phase1.log"

cmd = [
    sys.executable, "main_phase1.py",
    "--pretrained_weights", str(PTQ_CKPT),
    "--channel_wise",
    "--lambd", str(LAMBDA),
    "--epochs", str(EPOCHS_PHASE1),
    "--log", str(PHASE1_LOG),
]
print("running:", " ".join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)

# main_phase1.py writes the trained checkpoint next to the log file (.log -> .pth).
PHASE1_PTH = PHASE1_LOG.with_suffix(".pth")
print(f"phase 1 done — log: {PHASE1_LOG}\n              checkpoint: {PHASE1_PTH}")

running: /home/chang/anaconda3/envs/tram/bin/python main_phase1.py --pretrained_weights /home/chang/Work/Prev/TRAM-public/pretrained/cifar10_resnet18_w8a8_ptq.pth --channel_wise --lambd 0.3 --epochs 10 --log /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase1.log
log file: /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase1.log
tensorboard log path: /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase1.log_20260506_231504
Files already downloaded and verified
Files already downloaded and verified


Testing model: 100%|██████████| 40/40 [00:16<00:00,  2.47it/s]


phase 1 done — log: /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase1.log
              checkpoint: /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase1.pth


## 3. Phase 2 — AxM Structure Mapping

Phase 1 produces a vector of *continuous* structure parameters $\Theta^*$.
Phase 2 maps each $\Theta^{*(l)}$ to a *concrete* approximate multiplier netlist
by greedily replacing column compressors with constant-0 (Section 5 of the
paper). Three sub-steps, all wrapped in the cell below:

1. [`als-mult/find_best_structure.py`](als-mult/find_best_structure.py) parses
   the phase-1 log and emits a Verilog AxM netlist alongside it (`.v` next to
   the `.log`).
2. **Yosys** flattens the Verilog into a single-model BLIF
   (`sudo apt install yosys` if missing).
3. [`simulator/sim.py`](simulator/sim.py) exhaustively simulates the BLIF
   ($2^{16}$ patterns for an 8-bit AxM) and writes the forward LUT consumed by
   phase 3.

If you don't have Yosys handy, you can substitute one of the BLIFs already
shipped under [`app_mults/`](app_mults/) (e.g. `app_mults/OPACT/OPACT1.blif`)
and skip steps 1–2 — the resulting accuracy will track the corresponding
marker in Figure 4 (e.g. *OPACT_1*) instead of TRAM.

In [6]:
# Phase 2 — Verilog (find_best_structure.py) → BLIF (Yosys) → LUT (simulator/sim.py).
import shutil

PHASE2_VERILOG = PHASE1_LOG.with_suffix(".v")
PHASE2_BLIF    = RESULTS / f"resnet18_w8a8_lbd{LAMBDA}.blif"
PHASE2_LUT     = RESULTS / f"resnet18_w8a8_lbd{LAMBDA}.lut.txt"

# 4a. Continuous Θ → concrete Verilog AxM (writes alongside the phase-1 log)
subprocess.run(
    [sys.executable, "als-mult/find_best_structure.py",
     "--parse_file", str(PHASE1_LOG)],
    cwd=ROOT, check=True,
)
assert PHASE2_VERILOG.exists(), f"expected {PHASE2_VERILOG} from find_best_structure.py"

# 4b. Verilog → flat BLIF (requires yosys on PATH)
if shutil.which("yosys") is None:
    raise RuntimeError("yosys not found; install with `sudo apt install yosys`")
subprocess.run(
    ["yosys", "-p",
     f"read_verilog {PHASE2_VERILOG}; synth -flatten; write_blif {PHASE2_BLIF}"],
    check=True,
)

# 4c. BLIF → LUT (the file consumed by phase 3's --lut_file_name)
with open(PHASE2_LUT, "w") as fout:
    subprocess.run(
        [sys.executable, "simulator/sim.py", "--appMult", str(PHASE2_BLIF)],
        cwd=ROOT, stdout=fout, check=True,
    )
print(f"phase 2 done — Verilog: {PHASE2_VERILOG}\n              BLIF:    {PHASE2_BLIF}\n              LUT:     {PHASE2_LUT}")

parsed line: 2026-05-06 23:35:05,307 - layer_id = 20, gamma      = ['  1.000000', '  1.000000', '  1.000000', '  1.000000', '  0.174248', '  0.199563', '  0.194773', '  0.147112']
Output verilog file: /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase1.v
Target gamma values for 15 columns: 1.00000000, 1.00000000, 1.00000000, 1.00000000, 0.17424800, 0.19956300, 0.19477300, 0.14711200, 0.00000000, 0.00000000, 0.00000000, 0.00000000, 0.00000000, 0.00000000, 0.00000000
Building 8-bit array multiplier...
Total FAs: 46, Total HAs: 18
Simulation passed for 65536 patterns.
Verilog file written to: /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/Mult_8_8.v
MSE between accurate and target outputs: 9414.0000
Initial MSE after forcing p0_0=0: 9357.8623
zero all target signals in column 1 as target gamma is 1.0
zero all target signals in column 2 as target gamma is 1.0
zero all target signals in column 3 as target gamma is 1.0
skipping approxi

## 4. Phase 3 — Accuracy Recovery

Phase 3 freezes the AxM structure (the LUT from phase 2) and retrains the model
weights for **10 epochs** with cosine-annealed learning rate (5e-4 → 0). The
cell below shells out to [`main_phase3.py`](main_phase3.py) with the same
paper-faithful settings used in phase 1. After phase 3 the accuracy reported
in the log file should approach the corresponding $\lambda$-point on the TRAM
curve in Figure 4 (for ResNet18 / w8a8 / $\lambda=0.3$).

In [7]:
# Phase 3 — invoke main_phase3.py as a subprocess.
EPOCHS_PHASE3 = 10  # paper Section 6 setting
PHASE3_LOG = RESULTS / f"resnet18_w8a8_lbd{LAMBDA}_phase3.log"

cmd = [
    sys.executable, "main_phase3.py",
    "--pretrained_weights", str(PHASE1_PTH),
    "--channel_wise",
    "--lut_file_name", str(PHASE2_LUT),
    "--epochs", str(EPOCHS_PHASE3),
    "--log", str(PHASE3_LOG),
]
print("running:", " ".join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)
print(f"phase 3 done — final accuracy reported in {PHASE3_LOG}")

running: /home/chang/anaconda3/envs/tram/bin/python main_phase3.py --pretrained_weights /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase1.pth --channel_wise --lut_file_name /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3.lut.txt --epochs 10 --log /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase3.log
log file: /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase3.log
tensorboard log path: /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase3.log_20260506_233511
Files already downloaded and verified
Files already downloaded and verified
Initializing lookup tables from /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3.lut.txt...


Testing model: 100%|██████████| 40/40 [00:13<00:00,  2.95it/s]


phase 3 done — final accuracy reported in /home/chang/Work/Prev/TRAM-public/demo_resnet18_cifar10_results/resnet18_w8a8_lbd0.3_phase3.log


## 5. Bonus — Reproducing paper Table 1 with the simulator

Section 6.1 of the paper lists the **8-bit unsigned multipliers** used in the
experiments. Their `ER`, `NMED`, `MaxED` columns come from exhaustively
simulating each multiplier over all $2^{16}$ input combinations — exactly what
[`simulator/sim.py`](simulator/sim.py) does. The cell below verifies every entry.

In [8]:
import re
import subprocess

SIM = ROOT / "simulator" / "sim.py"

# (blif path relative to ROOT, expected ER%, expected NMED%, expected MaxED) from Table 1
EXPECTED = [
    ("app_mults/OPACT/OPACT1.blif",                   23.0,  0.022,  516),
    ("app_mults/OPACT/OPACTDesign18.blif",            51.0,  0.619, 7684),
    ("app_mults/evo_selected/mult8u/mul8u_0AB.blif",  97.7,  0.057,  115),
    ("app_mults/evo_selected/mult8u/mul8u_1DMU.blif", 66.0,  0.650, 4084),
    ("app_mults/evo_selected/mult8u/mul8u_GJM.blif",  74.9,  1.543, 9124),
]

row_fmt = "{:<46} {:>8} {:>10} {:>8}    {:>8} {:>10} {:>8}"
print(row_fmt.format("multiplier", "ER%", "NMED%", "MaxED", "← paper", "", ""))
print("-" * 110)
for blif, er, nmed, mxe in EXPECTED:
    out = subprocess.check_output(
        [sys.executable, str(SIM), "--appMult", str(ROOT / blif), "--mode", "full"],
        text=True)
    info = {}
    for line in out.splitlines():
        m = re.match(r"INFO: (.*?): (.+)", line)
        if m:
            info[m.group(1)] = m.group(2)
        if line.startswith("LUT for"):
            break
    er_actual   = float(info["Error rate"]) * 100
    nmed_actual = float(info["Normalized mean error distance"]) * 100
    mxe_actual  = int(info["Max error distance"])
    print(row_fmt.format(blif.split('/')[-1], f"{er_actual:.2f}", f"{nmed_actual:.3f}", f"{mxe_actual}", f"{er:.1f}", f"{nmed:.3f}", f"{mxe}"))
    assert abs(er_actual - er)   <= 0.05, f"{blif}: ER mismatch {er_actual} vs {er}"
    assert abs(nmed_actual - nmed) <= 0.001, f"{blif}: NMED mismatch {nmed_actual} vs {nmed}"
    assert mxe_actual == mxe, f"{blif}: MaxED mismatch {mxe_actual} vs {mxe}"
print("\nall five rows of Table 1 reproduced exactly.")

multiplier                                          ER%      NMED%    MaxED     ← paper                    
--------------------------------------------------------------------------------------------------------------
OPACT1.blif                                       23.03      0.022      516        23.0      0.022      516
OPACTDesign18.blif                                50.98      0.619     7684        51.0      0.619     7684
mul8u_0AB.blif                                    97.72      0.057      115        97.7      0.057      115
mul8u_1DMU.blif                                   65.97      0.650     4084        66.0      0.650     4084
mul8u_GJM.blif                                    74.91      1.543     9124        74.9      1.543     9124

all five rows of Table 1 reproduced exactly.
